In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
)

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_legal_assigns_proposals_arotron(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        # schema = (
        #        "LegalAssignsProposalsId string,"
        #        "SourceSystem string,"
        #        "PortfolioKey string,"
        #        "PortfolioName string,"
        #        "IngestionTimestamp timestamp,"
        #        "BatchId string,"
        #        "LegalAssignsProposalsKey string,"
        #        "CDI string,"
        #        "AgreementKey string,"
        #        "SubmittedDate date,"
        #        "ApprovalDate date,"
        #        "BusinessDate  date"
        # )
        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.LegalAssignsProposals"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                spark,
                curated_table_path,
                run_date,  # type: ignore
                "arotron",
            )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== ArxRecmd ==================
        ArxRecmd_extra_filters = None

        if ArxRecmd_extra_filters is not None:
            ArxRecmd_extra_filters.append(date_filter)
            ArxRecmd_final_filters = " AND ".join(ArxRecmd_extra_filters)
        else:
            ArxRecmd_final_filters = date_filter

            ArxRecmd_selected_fields = [
                "RcmdnID",
                "CustCDI",
                "ContNum",
                "RcmdSubmitDT",
                "RcmdApprvlDT",
                "BusinessDate",
            ]

            ArxRecmd = load_landed_delta_table(
                spark,
                catalog_landed,  # type: ignore
                "arotron_arxrecmd",
                ArxRecmd_final_filters,
                ArxRecmd_selected_fields,
            )

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== Joins ==================
        LegalAssignsProposals_df = ArxRecmd.join(
            portfolio,
            substring(ArxRecmd["CustCDI"], 1, 3) == portfolio["Prefix"],
            "left",
        )

        LegalAssignsProposals_primary_key_fields = [
            ArxRecmd.RcmdnID,
            lit("arotron"),
        ]

        LegalAssignsProposals_df = LegalAssignsProposals_df.withColumn(
            "LegalAssignsProposalsID",
            sha2(
                concat_ws(";", *LegalAssignsProposals_primary_key_fields),
                256,
            ),
        )

        legal_assigns_proposals_final_df = LegalAssignsProposals_df.select(
            LegalAssignsProposals_df.LegalAssignsProposalsID.cast("string"),
            lit("arotron").cast("String").alias("SourceSystem"),
            coalesce(portfolio.unit_code_profits, lit("-9998")).cast("string").alias("PortfolioKey"),
            coalesce(portfolio.PortfolioName, lit("Core Wholesale")).alias(
                "PortfolioName"
            ),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("String").alias("BatchId"),
            ArxRecmd.RcmdnID.cast("string").alias("LegalAssignsProposalsKey"),
            ArxRecmd.CustCDI.cast("string").alias("CDI"),
            ArxRecmd.ContNum.cast("string").alias("AgreementKey"),
            ArxRecmd.RcmdSubmitDT.cast("date").alias("SubmittedDate"),
            ArxRecmd.RcmdApprvlDT.cast("date").alias("ApprovalDate"),
            ArxRecmd.BusinessDate.cast("date").alias("BusinessDate"),
        )

        # ================== Append ==================

        legal_assigns_proposals_final_df.write.format("delta").clusterBy(
            "BusinessDate"
        ).mode("append").saveAsTable(curated_table_path)
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_legal_assigns_proposals_arotron(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)